# Decision explanation (M36)

Why the detector output a box, and why it missed one, at the float, integer and
golden stages. Each cell runs a command of `tools/explain.py` in a subprocess and
shows what it writes: PNG maps in `build/explain/maps/<net>/`, JSON tables in
`build/explain/`. The full protocol and the results are in
[results/explain.md](../results/explain.md); the interactive viewer is
`make explain`.

Cost (PC CPU, one process): Grad-CAM and saliency about 1 s per map, Integrated
Gradients 64 steps about 1 min, D-RISE 300 masks 1.5 min (float) to 3 min (integer).

Generated by `python -m tools.notebooks`: edit
`tools/notebooks/gabarits.py:explain`, not this file.

## Parameters

In [ ]:
NET     = 'tiny-yolov2-voc'          # tiny-yolov2-voc, tiny-yolov3-coco, tiny-yolov4-coco
IMAGES  = '000001,000004'            # VOC2007 test ids
METHODS = 'gradcam,saliency,lrp_int@int,lrp_float'  # method[@stage], see tools/explain.py
JOBS    = 4                          # processes of the table commands
N       = 10                         # images of the table commands (reduced runs)

## Environment

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

CANDIDATES = (Path.cwd(), *Path.cwd().parents, Path("/content/EmbeddedComputervision"))
ROOT = next((d for d in CANDIDATES if (d / "tools" / "explain.py").is_file()), None)
if ROOT is None:
    raise SystemExit(f"repository not found from {Path.cwd()}")
os.chdir(ROOT)


def run(*args):
    cmd = [sys.executable, "tools/explain.py", *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

## Maps of the best matched detection

One PNG per image: the image, then each method's map over it, the explained box in
white.

In [ ]:
run("map", "--nets", NET, "--image", IMAGES, "--methods", METHODS)
for i in IMAGES.split(","):
    p = ROOT / "build" / "explain" / "maps" / NET / f"{i}.png"
    if p.exists():
        display(Image(filename=str(p)))

## Integer attribution (T36.3)

Exact split of the accumulators (Σ qW·qx == acc − qb) and the relevance kept per
layer, then the tables of `results/explain.md` for the reduced run.

In [ ]:
run("intattr", "--nets", NET, "--n", str(N), "--jobs", str(JOBS))
run("report")
text = (ROOT / "results" / "explain.md").read_text()
a = text.index("<!-- explain:intattr:begin -->")
b = text.index("<!-- explain:intattr:end -->")
display(Markdown(text[a:b]))

## Miss causes of the persons (T36.5)

The cause of each missed person at the float and integer stages, by a fixed order of
tests (`yolo.explain.misses`).

In [ ]:
run("misses", "--nets", NET, "--n", str(5 * N), "--jobs", str(JOBS))
res = json.loads((ROOT / "build" / "explain" / "misses.json").read_text())[NET]
from collections import Counter
for st in ("float", "int"):
    print(st, dict(Counter(r[st] or "detected" for r in res)))